# Station Redundancy Analysis

Pairwise correlation, PCA, hierarchical clustering, and stanhope benchmarking
across all 6 PEINP/ECCC stations for four core meteorological variables.

**Stations**: cavendish, greenwich, north_rustico, stanhope (ECCC), stanley_bridge, tracadie

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, dendrogram
from pathlib import Path

from pea_met_network.redundancy import (
    build_station_matrix,
    pairwise_station_correlation,
    pca_station_loadings,
    cluster_station_order,
    benchmark_to_stanhope,
    build_station_recommendations,
)

STATIONS = ["cavendish", "greenwich", "north_rustico", "stanhope", "stanley_bridge", "tracadie"]
VARIABLES = {
    "air_temperature_c": "Air Temperature",
    "relative_humidity_pct": "Relative Humidity",
    "wind_speed_kmh": "Wind Speed",
    "rain_mm": "Precipitation",
}

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

In [2]:
# Load all station hourly data
frames = []
for s in STATIONS:
    f = Path("data/processed") / s / "station_hourly.csv"
    df = pd.read_csv(f, parse_dates=["timestamp_utc"], low_memory=False)
    df = df[["timestamp_utc", "air_temperature_c", "relative_humidity_pct",
             "wind_speed_kmh", "rain_mm"]].copy()
    df["station"] = s
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)
date_min = all_data["timestamp_utc"].min().date()
date_max = all_data["timestamp_utc"].max().date()
print(f"Loaded {len(all_data)} rows across {len(STATIONS)} stations")
print(f"Date range: {date_min} -> {date_max}")

Loaded 152489 rows across 6 stations
Date range: 2023-04-01 -> 2026-03-24


In [3]:
# Pre-compute matrices and analyses for all variables
results = {}
for col, label in VARIABLES.items():
    matrix = build_station_matrix(all_data, value_column=col)
    corr = pairwise_station_correlation(matrix)
    loadings = pca_station_loadings(matrix)
    order = cluster_station_order(matrix)
    bench = benchmark_to_stanhope(matrix, reference_station="stanhope")
    recs = build_station_recommendations(bench, pca_loadings=loadings, cluster_order=order)
    results[col] = {
        "label": label,
        "matrix": matrix,
        "corr": corr,
        "loadings": loadings,
        "order": order,
        "bench": bench,
        "recs": recs,
    }
    pc1_val = loadings[loadings['component']=='PC1']['explained_variance_ratio'].iloc[0]
    print(f"  {label}: {len(matrix)} hours, PC1={pc1_val:.3f}")
print("Done.")

  Air Temperature: 26113 hours, PC1=0.963


  Relative Humidity: 26113 hours, PC1=0.793


  Wind Speed: 26113 hours, PC1=0.709


  Precipitation: 26113 hours, PC1=0.547
Done.


## 1. Correlation Heatmaps

Pairwise Pearson correlation for each variable across all 6 stations.
Values close to 1.0 indicate redundant signals; values < 0.7 indicate independent information.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
fig.suptitle("Pairwise Station Correlation by Variable", fontsize=14, fontweight="bold", y=0.98)

cmap = LinearSegmentedColormap.from_list(
    "corr", [(0.85, 0.15, 0.15), (0.95, 0.95, 0.95), (0.15, 0.35, 0.85)], N=256
)

for idx, (col, label) in enumerate(VARIABLES.items()):
    ax = axes[idx // 2][idx % 2]
    corr = results[col]["corr"]
    order = results[col]["order"]
    corr_sorted = corr.loc[order, order]
    im = ax.imshow(corr_sorted, cmap=cmap, vmin=0.0, vmax=1.0, aspect="equal")

    for i in range(len(order)):
        for j in range(len(order)):
            val = corr_sorted.iloc[i, j]
            color = "white" if val < 0.5 or val > 0.9 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color=color, fontweight="bold")

    ax.set_xticks(range(len(order)))
    ax.set_yticks(range(len(order)))
    short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ") for s in order]
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(short, fontsize=8)
    ax.set_title(label, fontsize=11)

    pc1 = results[col]["loadings"][results[col]["loadings"]["component"]=="PC1"]["explained_variance_ratio"].iloc[0]
    ax.text(0.02, 0.02, f"PC1 = {pc1:.1%}", transform=ax.transAxes,
            fontsize=8, color="0.3", style="italic")

fig.colorbar(im, ax=axes, shrink=0.6, label="Pearson r")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(OUTPUT_DIR / "redundancy_correlation_heatmaps.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_correlation_heatmaps.png")

/tmp/ipykernel_215571/250028434.py:34: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.95])
findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=8.0, and fallback to the default font was disabled

## 2. Hierarchical Clustering Dendrograms

Agglomerative clustering (average linkage) on distance = 1 - r.
Stations that merge early are the most redundant.

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Station Clustering Dendrograms (distance = 1 - r)", fontsize=14, fontweight="bold", y=0.98)

for idx, (col, label) in enumerate(VARIABLES.items()):
    ax = axes[idx // 2][idx % 2]
    corr = results[col]["corr"]
    distance = 1.0 - corr
    from scipy.spatial.distance import squareform
    dist_condensed = squareform(distance.values, checks=False)
    Z = linkage(dist_condensed, method="average", metric="euclidean")
    short_labels = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ") for s in corr.columns]
    dendrogram(Z, labels=short_labels, ax=ax, leaf_rotation=45, leaf_font_size=8, color_threshold=0.0)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel("Distance (1 - r)")
    ax.grid(axis="y", alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(OUTPUT_DIR / "redundancy_dendrograms.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_dendrograms.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=8.0, and fallback to the default font was disabled

## 3. Temperature Scatter - The Tight Cluster

Cavendish, north rustico, and tracadie have pairwise r >= 0.994 for temperature.
These scatter plots show how tightly coupled they are.

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Temperature: Cavendish - North Rustico - Tracadie Cluster", fontsize=13, fontweight="bold")

matrix = results["air_temperature_c"]["matrix"]
pairs = [("cavendish", "north_rustico"), ("cavendish", "tracadie"), ("north_rustico", "tracadie")]

for ax, (s1, s2) in zip(axes, pairs):
    valid = matrix[[s1, s2]].dropna()
    ax.scatter(valid[s1], valid[s2], s=1, alpha=0.15, c="steelblue", rasterized=True)
    lo = min(valid[s1].min(), valid[s2].min())
    hi = max(valid[s1].max(), valid[s2].max())
    ax.plot([lo, hi], [lo, hi], "--", color="red", alpha=0.6, linewidth=1)
    r = valid[s1].corr(valid[s2])
    mad = (valid[s1] - valid[s2]).abs().mean()
    n = len(valid)
    short1 = s1.replace("north_", "N.").replace("_", " ").title()
    short2 = s2.replace("north_", "N.").replace("_", " ").title()
    ax.set_xlabel(f"{short1}")
    ax.set_ylabel(f"{short2}")
    ax.set_title(f"r = {r:.3f}  |  MAD = {mad:.2f}  |  n = {n:,}", fontsize=10)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_temp_scatter_cluster.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_temp_scatter_cluster.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

## 4. Temperature Scatter - All Stations vs Stanhope

Every PEINP station plotted against the ECCC stanhope reference.

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Temperature: All Stations vs Stanhope (ECCC Reference)", fontsize=13, fontweight="bold")

matrix = results["air_temperature_c"]["matrix"]
peinp_stations = [s for s in STATIONS if s != "stanhope"]

for ax, station in zip(axes.flat, peinp_stations):
    valid = matrix[[station, "stanhope"]].dropna()
    ax.scatter(valid["stanhope"], valid[station], s=1, alpha=0.12, c="steelblue", rasterized=True)
    lo = min(valid["stanhope"].min(), valid[station].min())
    hi = max(valid["stanhope"].max(), valid[station].max())
    ax.plot([lo, hi], [lo, hi], "--", color="red", alpha=0.5, linewidth=1)
    r = valid["stanhope"].corr(valid[station])
    mad = (valid["stanhope"] - valid[station]).abs().mean()
    n = len(valid)
    short = station.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ").title()
    ax.set_xlabel("Stanhope", fontsize=9)
    ax.set_ylabel(f"{short}", fontsize=9)
    ax.set_title(f"{short}: r={r:.3f}, MAD={mad:.2f}, n={n:,}", fontsize=9)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_temp_scatter_vs_stanhope.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_temp_scatter_vs_stanhope.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

## 5. Stanhope Benchmark - Mean Absolute Difference

How far each station deviates from stanhope on average, per variable.

In [8]:
# Build benchmark summary across all variables
bench_summary = []
for col, label in VARIABLES.items():
    b = results[col]["bench"].copy()
    b["variable"] = label
    bench_summary.append(b[["station", "variable", "mean_abs_diff", "correlation", "overlap_count"]])

bench_df = pd.concat(bench_summary, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Stanhope Benchmark: Station Deviation by Variable", fontsize=13, fontweight="bold")

peinp_stations_sorted = [s for s in STATIONS if s != "stanhope"]
short_names = [s.replace("north_", "N.\n").replace("stanley_", "S.\n").replace("_", " ").title()
                for s in peinp_stations_sorted]

ax = axes[0]
x = np.arange(len(peinp_stations_sorted))
width = 0.18
colors = ["#2166ac", "#4393c3", "#92c5de", "#d1e5f0"]

for i, (col, label) in enumerate(VARIABLES.items()):
    vals = bench_df[bench_df["variable"] == label].set_index("station").loc[peinp_stations_sorted, "mean_abs_diff"]
    matrix = results[col]["matrix"]
    var_range = matrix.max().max() - matrix.min().min()
    vals_pct = (vals / var_range * 100).values
    ax.bar(x + i * width, vals_pct, width, label=label, color=colors[i], edgecolor="white", linewidth=0.5)

ax.set_xlabel("Station")
ax.set_ylabel("Mean Abs Difference (% of range)")
ax.set_title("Deviation from Stanhope (normalized)", fontsize=11)
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(short_names, fontsize=8)
ax.legend(fontsize=7, loc="upper left")
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
for i, (col, label) in enumerate(VARIABLES.items()):
    vals = bench_df[bench_df["variable"] == label].set_index("station").loc[peinp_stations_sorted, "correlation"]
    ax.bar(x + i * width, vals.values, width, label=label, color=colors[i], edgecolor="white", linewidth=0.5)

ax.set_xlabel("Station")
ax.set_ylabel("Correlation with Stanhope (r)")
ax.set_title("Correlation with Stanhope by Variable", fontsize=11)
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(short_names, fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.95, color="red", linestyle="--", alpha=0.4, linewidth=0.8, label="r = 0.95")
ax.legend(fontsize=7, loc="lower left")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_benchmark_bars.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_benchmark_bars.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=8.0, and fallback to the default font was disabled

## 6. Summary: Recommendation Matrix

Keep/remove/defer recommendation per station per variable,
with risk band from uncertainty quantification.

In [9]:
# Build recommendation matrix
rec_rows = []
for col, label in VARIABLES.items():
    recs = results[col]["recs"]
    bench = results[col]["bench"]
    for _, row in recs.iterrows():
        b_row = bench[bench["station"] == row["station"]].iloc[0]
        rec_rows.append({
            "station": row["station"],
            "variable": label,
            "recommendation": row["recommendation"],
            "risk_band": row["risk_band"],
            "risk_prob": row["risk_probability"],
            "correlation": b_row["correlation"],
            "mad": b_row["mean_abs_diff"],
        })

rec_df = pd.DataFrame(rec_rows)

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("Redundancy Recommendations by Station and Variable", fontsize=13, fontweight="bold")

rec_color_map = {"keep": "#2ca02c", "defer": "#ff7f0e", "remove": "#d62728"}

pivot = rec_df.pivot_table(index="station", columns="variable", values="recommendation", aggfunc="first")
pivot = pivot.loc[peinp_stations_sorted]

for i, station in enumerate(peinp_stations_sorted):
    for j, label in enumerate(VARIABLES.values()):
        rec = pivot.loc[station, label]
        risk = rec_df[(rec_df["station"] == station) & (rec_df["variable"] == label)]["risk_band"].iloc[0]
        corr = rec_df[(rec_df["station"] == station) & (rec_df["variable"] == label)]["correlation"].iloc[0]
        color = rec_color_map[rec]
        ax.barh(i, 1, left=j, color=color, edgecolor="white", linewidth=1.5, height=0.7)
        ax.text(j + 0.5, i, f"{rec}\nr={corr:.2f}", ha="center", va="center",
                fontsize=8, fontweight="bold", color="white" if rec == "remove" else "black")

ax.set_yticks(range(len(peinp_stations_sorted)))
short_y = [s.replace("north_", "N. ").replace("stanley_", "S. ").replace("_", " ").title()
           for s in peinp_stations_sorted]
ax.set_yticklabels(short_y)
ax.set_xticks([j + 0.5 for j in range(len(VARIABLES))])
ax.set_xticklabels([l for l in VARIABLES.values()], rotation=20, ha="right", fontsize=9)
ax.set_xlim(0, len(VARIABLES))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=rec_color_map[k], edgecolor="white", label=k.capitalize())
                   for k in ["keep", "defer", "remove"]]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9, title="Recommendation")
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_recommendations.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_recommendations.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=9.0, and fallback to the default font was disabled

## 7. Output File Listing

All figures generated by this notebook.

In [10]:
from pathlib import Path
figs = sorted(OUTPUT_DIR.glob("redundancy_*.png"))
for f in figs:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s} {size_kb:>6.0f} KB")
print(f"\n  {len(figs)} figures total")

  redundancy_benchmark_bars.png                          92 KB
  redundancy_bootstrap_ci_forest.png                    217 KB
  redundancy_bootstrap_ci_heatmap.png                   130 KB
  redundancy_correlation_heatmaps.png                   243 KB
  redundancy_dendrograms.png                            114 KB
  redundancy_extreme_degradation.png                    108 KB
  redundancy_extreme_vs_full.png                        183 KB
  redundancy_fire_weather_corr.png                      155 KB
  redundancy_fwi_correlation_grid.png                   262 KB
  redundancy_fwi_dendrograms.png                        125 KB
  redundancy_fwi_recommendations.png                     77 KB
  redundancy_loo_degradation.png                         90 KB
  redundancy_loo_summary.png                             55 KB
  redundancy_partial_corr_scatter.png                   206 KB
  redundancy_recommendations.png                         91 KB
  redundancy_regional_dependence.png                   

## 8. Bootstrap Confidence Intervals on Correlations

Block bootstrap (24-hour blocks, 10 000 iterations) to quantify uncertainty
around each pairwise Pearson *r*.  Stations whose lower 95% CI stays above
the 0.95 removal threshold are *confidently* redundant.

In [11]:
from scipy.spatial.distance import squareform

BLOCK_SIZE = 24
N_BOOTSTRAP = 10_000
THRESHOLD = 0.95

bootstrap_results = {}
rng = np.random.default_rng(42)

for col, label in VARIABLES.items():
    matrix = results[col]["matrix"].dropna(axis="index", how="any")
    n = len(matrix)
    stations_list = list(matrix.columns)
    n_st = len(stations_list)
    rows = []

    for i in range(n_st):
        for j in range(i + 1, n_st):
            s1, s2 = stations_list[i], stations_list[j]
            xy = matrix[[s1, s2]].dropna().values
            n_valid = len(xy)
            r_hat = np.corrcoef(xy[:, 0], xy[:, 1])[0, 1]

            r_boot = np.empty(N_BOOTSTRAP)
            for b in range(N_BOOTSTRAP):
                block_idx = rng.integers(0, n_valid // BLOCK_SIZE, size=n_valid // BLOCK_SIZE)
                indices = np.concatenate([np.arange(bl * BLOCK_SIZE, (bl + 1) * BLOCK_SIZE) for bl in block_idx])
                sample = xy[indices[:n_valid]]
                r_boot[b] = np.corrcoef(sample[:, 0], sample[:, 1])[0, 1]

            ci_lo, ci_hi = np.percentile(r_boot, [2.5, 97.5])
            confident = ci_lo >= THRESHOLD
            rows.append({
                "station_i": s1, "station_j": s2,
                "r_hat": r_hat, "ci_lo": ci_lo, "ci_hi": ci_hi,
                "confidently_redundant": confident,
            })
            mark = "YES" if confident else "no"
            print(f"  {label[:12]:12s} {s1:>15s} vs {s2:>15s}: r={r_hat:.3f}  95% CI=[{ci_lo:.3f}, {ci_hi:.3f}]  {mark}")

    bootstrap_results[col] = pd.DataFrame(rows)

print("Bootstrap complete.")

  Air Temperat       cavendish vs       greenwich: r=0.930  95% CI=[0.895, 0.964]  no


  Air Temperat       cavendish vs   north_rustico: r=0.995  95% CI=[0.994, 0.995]  YES


  Air Temperat       cavendish vs        stanhope: r=0.969  95% CI=[0.966, 0.972]  YES


  Air Temperat       cavendish vs  stanley_bridge: r=0.968  95% CI=[0.947, 0.987]  no


  Air Temperat       cavendish vs        tracadie: r=0.994  95% CI=[0.993, 0.994]  YES


...

  Precipitatio   north_rustico vs  stanley_bridge: r=0.658  95% CI=[0.601, 0.722]  no


  Precipitatio   north_rustico vs        tracadie: r=0.589  95% CI=[0.505, 0.667]  no


  Precipitatio        stanhope vs  stanley_bridge: r=0.234  95% CI=[0.185, 0.287]  no


  Precipitatio        stanhope vs        tracadie: r=0.166  95% CI=[0.117, 0.225]  no


  Precipitatio  stanley_bridge vs        tracadie: r=0.504  95% CI=[0.436, 0.578]  no
Bootstrap complete.


In [12]:
# --- Visual: Bootstrap CI Forest Plot ---
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Bootstrap 95% Confidence Intervals - Pairwise Correlations", fontsize=13, fontweight="bold", y=0.99)

for idx, (col, label) in enumerate(VARIABLES.items()):
    ax = axes[idx // 2][idx % 2]
    df = bootstrap_results[col].copy()
    df["pair"] = df["station_i"] + " - " + df["station_j"]
    df["pair"] = df["pair"].str.replace("north_", "N.").str.replace("stanley_", "S.").str.replace("_", " ")
    df = df.sort_values("r_hat", ascending=True)

    y = range(len(df))
    colors = ["#2ca02c" if c else "#d62728" for c in df["confidently_redundant"]]
    ax.barh(y, df["r_hat"] - df["ci_lo"], left=df["ci_lo"], height=0.5,
            color=colors, alpha=0.25, edgecolor="none")
    ax.scatter(df["r_hat"], y, s=25, zorder=3, c=colors, edgecolors="white", linewidths=0.4)
    ax.axvline(THRESHOLD, color="red", ls="--", lw=1, alpha=0.6)
    ax.set_yticks(y)
    ax.set_yticklabels(df["pair"], fontsize=7)
    ax.set_xlabel("Pearson r")
    ax.set_title(label, fontsize=10)
    ax.set_xlim(0.3, 1.02)
    ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ca02c", alpha=0.5, label="CI entirely above 0.95"),
    Patch(facecolor="#d62728", alpha=0.5, label="CI crosses 0.95"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.03, 1, 0.97])
fig.savefig(OUTPUT_DIR / "redundancy_bootstrap_ci_forest.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_bootstrap_ci_forest.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

In [13]:
# --- Visual: Bootstrap CI Heatmap (lower / upper) for temperature ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Bootstrap 95% CI Range by Station Pair - Temperature", fontsize=13, fontweight="bold")

col = "air_temperature_c"
df = bootstrap_results[col]
stations_all = list(dict.fromkeys(list(df["station_i"]) + list(df["station_j"])))
n = len(stations_all)
lo_mat = np.full((n, n), np.nan)
hi_mat = np.full((n, n), np.nan)

for _, row in df.iterrows():
    i = stations_all.index(row["station_i"])
    j = stations_all.index(row["station_j"])
    lo_mat[i, j] = lo_mat[j, i] = row["ci_lo"]
    hi_mat[i, j] = hi_mat[j, i] = row["ci_hi"]

short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ") for s in stations_all]

for ax, mat, title in [(axes[0], lo_mat, "Lower 95% CI"), (axes[1], hi_mat, "Upper 95% CI")]:
    im = ax.imshow(mat, cmap="RdBu", vmin=0.5, vmax=1.0, aspect="equal")
    for i in range(n):
        for j in range(n):
            if not np.isnan(mat[i, j]):
                color = "white" if mat[i, j] < 0.7 or mat[i, j] > 0.9 else "black"
                ax.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center", fontsize=8, color=color)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(short, fontsize=8)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_bootstrap_ci_heatmap.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_bootstrap_ci_heatmap.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=8.0, and fallback to the default font was disabled

## 9. Partial Correlation - Controlling for Stanhope

Strips out the shared regional weather signal by conditioning on stanhope
(the ECCC reference).  If partial *r* collapses, stations merely share an
airmass; if it stays high, they are genuinely locally redundant.

In [14]:
def partial_correlation(a, b, c):
    valid = pd.concat([a, b, c], axis=1).dropna()
    a_v, b_v, c_v = valid[a.name], valid[b.name], valid[c.name]
    coefs_a = np.polyfit(c_v, a_v, 1)
    coefs_b = np.polyfit(c_v, b_v, 1)
    resid_a = a_v - np.polyval(coefs_a, c_v)
    resid_b = b_v - np.polyval(coefs_b, c_v)
    return np.corrcoef(resid_a, resid_b)[0, 1]

partial_results = {}
peinp = [s for s in STATIONS if s != "stanhope"]

for col, label in VARIABLES.items():
    matrix = results[col]["matrix"]
    rows = []
    for i, s1 in enumerate(peinp):
        for j, s2 in enumerate(peinp):
            if j <= i:
                continue
            r_full = matrix[s1].corr(matrix[s2])
            r_partial = partial_correlation(matrix[s1], matrix[s2], matrix["stanhope"])
            if abs(r_full) > 1e-9:
                regional_dep = 1.0 - (r_partial / r_full)
            else:
                regional_dep = np.nan
            rows.append({
                "station_i": s1, "station_j": s2,
                "r_marginal": r_full, "r_partial": r_partial,
                "regional_dependence": regional_dep,
            })
            print(f"  {label[:12]:12s} {s1:>15s} vs {s2:>15s}: marginal={r_full:.3f}  partial={r_partial:.3f}  reg_dep={regional_dep:.1%}")
    partial_results[col] = pd.DataFrame(rows)

print("Partial correlation complete.")

  Air Temperat       cavendish vs       greenwich: marginal=0.907  partial=0.321  reg_dep=64.6%
  Air Temperat       cavendish vs   north_rustico: marginal=0.994  partial=0.914  reg_dep=8.0%
  Air Temperat       cavendish vs  stanley_bridge: marginal=0.972  partial=0.718  reg_dep=26.1%
  Air Temperat       cavendish vs        tracadie: marginal=0.994  partial=0.897  reg_dep=9.8%
  Air Temperat       greenwich vs   north_rustico: marginal=0.914  partial=0.321  reg_dep=64.8%
  Air Temperat       greenwich vs  stanley_bridge: marginal=0.898  partial=0.261  reg_dep=71.0%
  Air Temperat       greenwich vs        tracadie: marginal=0.942  partial=0.410  reg_dep=56.5%
  Air Temperat   north_rustico vs  stanley_bridge: marginal=0.974  partial=0.690  reg_dep=29.2%
  Air Temperat   north_rustico vs        tracadie: marginal=0.995  partial=0.915  reg_dep=8.1%
  Air Temperat  stanley_bridge vs        tracadie: marginal=0.974  partial=0.669  reg_dep=31.3%
  Relative Hum       cavendish vs       gre

In [15]:
# --- Visual: Marginal vs Partial r scatter ---
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle("Marginal vs Partial Correlation (controlling for Stanhope)", fontsize=13, fontweight="bold", y=0.99)

for idx, (col, label) in enumerate(VARIABLES.items()):
    ax = axes[idx // 2][idx % 2]
    df = partial_results[col]
    ax.scatter(df["r_marginal"], df["r_partial"], s=60, c="steelblue", edgecolors="white",
               linewidths=0.6, zorder=3, alpha=0.85)
    lo = min(df["r_marginal"].min(), df["r_partial"].min()) - 0.05
    hi = max(df["r_marginal"].max(), df["r_partial"].max()) + 0.05
    ax.plot([lo, hi], [lo, hi], "--", color="grey", alpha=0.5, lw=1)

    for _, row in df.iterrows():
        short_i = row["station_i"].replace("north_", "N.").replace("stanley_", "S.").replace("_", " ")
        short_j = row["station_j"].replace("north_", "N.").replace("stanley_", "S.").replace("_", " ")
        ax.annotate(f"{short_i}-{short_j}", (row["r_marginal"], row["r_partial"]),
                    fontsize=5.5, ha="left", va="bottom", xytext=(3, 3), textcoords="offset points")

    ax.set_xlabel("Marginal r (full)")
    ax.set_ylabel("Partial r (| stanhope)")
    ax.set_title(label, fontsize=10)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUTPUT_DIR / "redundancy_partial_corr_scatter.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_partial_corr_scatter.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

In [16]:
# --- Visual: Regional Dependence Heatmap (temperature) ---
fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle("Regional Dependence Fraction - Temperature\n(fraction of correlation explained by shared regional weather)",
             fontsize=12, fontweight="bold")

col = "air_temperature_c"
df = partial_results[col]
stations_peinp = list(dict.fromkeys(list(df["station_i"]) + list(df["station_j"])))
n = len(stations_peinp)
dep_mat = np.eye(n)

for _, row in df.iterrows():
    i = stations_peinp.index(row["station_i"])
    j = stations_peinp.index(row["station_j"])
    dep_mat[i, j] = dep_mat[j, i] = row["regional_dependence"]

short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ").title() for s in stations_peinp]

cmap = LinearSegmentedColormap.from_list("dep", ["#2166ac", "white", "#d62728"], N=256)
im = ax.imshow(dep_mat, cmap=cmap, vmin=0, vmax=1, aspect="equal")
for i in range(n):
    for j in range(n):
        color = "white" if dep_mat[i, j] > 0.7 or dep_mat[i, j] < 0.2 else "black"
        ax.text(j, i, f"{dep_mat[i,j]:.0%}", ha="center", va="center", fontsize=9,
                color=color, fontweight="bold")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short, fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8, label="Regional Dependence Fraction")
ax.grid(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_regional_dependence.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_regional_dependence.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=9.0, and fallback to the default font was disabled

## 10. Extreme Event Correlation

Stations agree on calm days - the real question is whether they agree when
conditions are dangerous.  Correlation is computed only on hours where
**either** station exceeds the 95th or 99th percentile.  FWI fire-weather
days (FWI >= 15, FWI >= 25) are analysed separately.

In [17]:
# Load FWI columns for Plans 2 and 1
fwi_frames = []
for s in STATIONS:
    f = Path("data/processed") / s / "station_hourly.csv"
    df = pd.read_csv(f, parse_dates=["timestamp_utc"], low_memory=False)
    fwi_cols = ["timestamp_utc", "ffmc", "dmc", "dc", "isi", "bui", "fwi"]
    available = [c for c in fwi_cols if c in df.columns]
    df_sub = df[available].copy()
    df_sub["station"] = s
    fwi_frames.append(df_sub)

all_data_fwi = pd.concat(fwi_frames, ignore_index=True)
print(f"FWI data loaded: {len(all_data_fwi)} rows across {len(STATIONS)} stations")

FWI data loaded: 152489 rows across 6 stations


In [18]:
PERCENTILES = [0.95, 0.99]
FWI_THRESHOLDS = [15, 25]
extreme_results = []

for col, label in VARIABLES.items():
    matrix = results[col]["matrix"]
    combined = matrix.values.flatten()
    combined = combined[~np.isnan(combined)]

    for pct in PERCENTILES:
        threshold = np.nanpercentile(combined, pct * 100)
        for i, s1 in enumerate(list(matrix.columns)):
            for j, s2 in enumerate(list(matrix.columns)):
                if j <= i:
                    continue
                pair = matrix[[s1, s2]].dropna()
                r_full = pair[s1].corr(pair[s2])
                extreme_mask = (pair[s1] >= threshold) | (pair[s2] >= threshold)
                extreme_pair = pair[extreme_mask]
                if len(extreme_pair) < 30:
                    r_extreme = np.nan
                else:
                    r_extreme = extreme_pair[s1].corr(extreme_pair[s2])
                delta = r_extreme - r_full if not np.isnan(r_extreme) else np.nan
                extreme_results.append({
                    "variable": label, "station_i": s1, "station_j": s2,
                    "percentile": f"{int(pct*100)}th", "threshold": threshold,
                    "r_full": r_full, "r_extreme": r_extreme, "delta_r": delta,
                    "n_extreme": len(extreme_pair),
                })

# FWI fire-weather days
FWI_VARS = {"fwi": "FWI", "ffmc": "FFMC", "isi": "ISI"}
for fwicol, fwilabel in FWI_VARS.items():
    fwimat = build_station_matrix(all_data_fwi, value_column=fwicol)
    for thr in FWI_THRESHOLDS:
        for i, s1 in enumerate(list(fwimat.columns)):
            for j, s2 in enumerate(list(fwimat.columns)):
                if j <= i:
                    continue
                pair = fwimat[[s1, s2]].dropna()
                r_full = pair[s1].corr(pair[s2])
                fire_mask = (pair[s1] >= thr) | (pair[s2] >= thr)
                fire_pair = pair[fire_mask]
                if len(fire_pair) < 30:
                    r_extreme = np.nan
                else:
                    r_extreme = fire_pair[s1].corr(fire_pair[s2])
                delta = r_extreme - r_full if not np.isnan(r_extreme) else np.nan
                extreme_results.append({
                    "variable": fwilabel, "station_i": s1, "station_j": s2,
                    "percentile": f"FWI>={thr}", "threshold": thr,
                    "r_full": r_full, "r_extreme": r_extreme, "delta_r": delta,
                    "n_extreme": len(fire_pair),
                })

extreme_df = pd.DataFrame(extreme_results)
print(f"Extreme correlation computed: {len(extreme_df)} pairs.")
print(extreme_df.groupby(["variable", "percentile"])["delta_r"].mean().to_string())

/mnt/fast_data/workspaces/pea-met-network/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/fast_data/workspaces/pea-met-network/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/fast_data/workspaces/pea-met-network/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/fast_data/workspaces/pea-met-network/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Extreme correlation computed: 210 pairs.
variable           percentile
Air Temperature    95th         -0.803835
                   99th         -1.070063
FFMC               FWI>=15      -0.055471
                   FWI>=25      -0.103207
FWI                FWI>=15      -0.791045
                   FWI>=25      -1.139844
ISI                FWI>=15      -0.847521
                   FWI>=25            NaN
Precipitation      95th         -0.214687
                   99th         -0.540212
Relative Humidity  95th         -0.980790
                   99th         -1.008797
Wind Speed         95th         -0.707902
                   99th         -0.928183


In [19]:
# --- Visual: Extreme vs Full-record Correlation ---
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Extreme vs Full-Record Correlation", fontsize=13, fontweight="bold", y=0.99)

for idx, (col, label) in enumerate(VARIABLES.items()):
    ax = axes[idx // 2][idx % 2]
    sub = extreme_df[(extreme_df["variable"] == label) & (extreme_df["percentile"] == "95th")].copy()
    sub["pair"] = (sub["station_i"].str.replace("north_", "N.").str.replace("stanley_", "S.").str.replace("_", " ") +
                 " - " + sub["station_j"].str.replace("north_", "N.").str.replace("stanley_", "S.").str.replace("_", " "))
    sub = sub.sort_values("r_full", ascending=True)

    x = np.arange(len(sub))
    w = 0.35
    ax.barh(x - w/2, sub["r_full"], w, label="Full record", color="#4393c3", alpha=0.85)
    ax.barh(x + w/2, sub["r_extreme"], w, label="95th pctl extremes", color="#d6604d", alpha=0.85)

    ax.set_yticks(x)
    ax.set_yticklabels(sub["pair"], fontsize=6.5)
    ax.set_xlabel("Pearson r")
    ax.set_title(label, fontsize=10)
    ax.set_xlim(0, 1.05)
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(axis="x", alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUTPUT_DIR / "redundancy_extreme_vs_full.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_extreme_vs_full.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

In [20]:
# --- Visual: Extreme Degradation Heatmap (temperature) ---
fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle("Extreme Correlation Degradation - Temperature (95th percentile)\ndelta_r = r_extreme - r_full  (negative = worse on extremes)",
             fontsize=11, fontweight="bold")

sub = extreme_df[(extreme_df["variable"] == "Air Temperature") & (extreme_df["percentile"] == "95th")]
stations_all = list(dict.fromkeys(list(sub["station_i"]) + list(sub["station_j"])))
n = len(stations_all)
deg_mat = np.zeros((n, n))

for _, row in sub.iterrows():
    i = stations_all.index(row["station_i"])
    j = stations_all.index(row["station_j"])
    deg_mat[i, j] = deg_mat[j, i] = row["delta_r"]

short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ").title() for s in stations_all]

cmap = LinearSegmentedColormap.from_list("deg", ["#d73027", "#fff7ec", "#1a9850"], N=256)
im = ax.imshow(deg_mat, cmap=cmap, vmin=-0.15, vmax=0.05, aspect="equal")
for i in range(n):
    for j in range(n):
        color = "white" if abs(deg_mat[i, j]) > 0.08 else "black"
        ax.text(j, i, f"{deg_mat[i,j]:+.3f}", ha="center", va="center", fontsize=9,
                color=color, fontweight="bold")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short, fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8, label="delta_r (extreme - full)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_extreme_degradation.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_extreme_degradation.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=9.0, and fallback to the default font was disabled

In [21]:
# --- Visual: Fire Weather Day Correlation (FWI >= 15) ---
fig, ax = plt.subplots(figsize=(10, 7))
fig.suptitle("Correlation on Fire Weather Days (FWI >= 15)", fontsize=13, fontweight="bold")

sub = extreme_df[extreme_df["percentile"] == "FWI>=15"].copy()
if len(sub) > 0:
    sub["pair"] = (sub["station_i"].str.replace("north_", "N.").str.replace("stanley_", "S.").str.replace("_", " ") +
                 " - " + sub["station_j"].str.replace("north_", "N.").str.replace("stanley_", "S.").str.replace("_", " "))
    sub = sub.sort_values("r_extreme", ascending=True)

    y = np.arange(len(sub))
    colors = ["#d62728" if r < 0.8 else "#ff7f0e" if r < 0.9 else "#2ca02c" for r in sub["r_extreme"]]
    ax.barh(y, sub["r_extreme"], height=0.6, color=colors, alpha=0.85, edgecolor="white")
    ax.set_yticks(y)
    ax.set_yticklabels(sub["pair"], fontsize=7)
    ax.set_xlabel("Pearson r on FWI >= 15 days")
    ax.axvline(0.95, color="red", ls="--", lw=1, alpha=0.5)
    ax.axvline(0.80, color="orange", ls="--", lw=0.8, alpha=0.4)
    ax.set_xlim(0, 1.05)
    ax.grid(axis="x", alpha=0.3)
else:
    ax.text(0.5, 0.5, "No FWI >= 15 pairs found", ha="center", va="center", transform=ax.transAxes)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_fire_weather_corr.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_fire_weather_corr.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

## 11. FWI-Component Redundancy

Run the full correlation / PCA / clustering suite on the six FWI outputs
(FFMC, DMC, DC, ISI, BUI, FWI) instead of raw meteorological inputs.
DC and DMC are daily-integrated (slow response) and should cluster tightly;
FFMC is hourly (fast response) and may diverge more.

In [22]:
FWI_COMPONENTS = {"ffmc": "FFMC", "dmc": "DMC", "dc": "DC", "isi": "ISI", "bui": "BUI", "fwi": "FWI"}

fwi_results = {}

for fwicol, fwilabel in FWI_COMPONENTS.items():
    matrix = build_station_matrix(all_data_fwi, value_column=fwicol)
    corr = pairwise_station_correlation(matrix)
    loadings = pca_station_loadings(matrix)
    order = cluster_station_order(matrix)
    bench = benchmark_to_stanhope(matrix, reference_station="stanhope")
    recs = build_station_recommendations(bench, pca_loadings=loadings, cluster_order=order)
    fwi_results[fwicol] = {
        "label": fwilabel, "matrix": matrix, "corr": corr,
        "loadings": loadings, "order": order, "bench": bench, "recs": recs,
    }
    pc1 = loadings[loadings["component"]=="PC1"]["explained_variance_ratio"].iloc[0]
    print(f"  {fwilabel:4s}: {len(matrix)} hours, PC1={pc1:.3f}")

print("FWI component analysis complete.")

  FFMC: 26093 hours, PC1=0.854
  DMC : 26110 hours, PC1=0.932
  DC  : 26110 hours, PC1=0.967


  ISI : 26090 hours, PC1=0.661
  BUI : 26090 hours, PC1=0.943
  FWI : 26090 hours, PC1=0.872
FWI component analysis complete.


In [23]:
# --- Visual: FWI Correlation Heatmap Grid (2x3) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("FWI Component Correlation by Station Pair", fontsize=14, fontweight="bold", y=0.99)

cmap = LinearSegmentedColormap.from_list(
    "corr", [(0.85, 0.15, 0.15), (0.95, 0.95, 0.95), (0.15, 0.35, 0.85)], N=256
)

for idx, (fwicol, fwilabel) in enumerate(FWI_COMPONENTS.items()):
    ax = axes[idx // 3][idx % 3]
    corr = fwi_results[fwicol]["corr"]
    order = fwi_results[fwicol]["order"]
    corr_sorted = corr.loc[order, order]
    im = ax.imshow(corr_sorted, cmap=cmap, vmin=0.0, vmax=1.0, aspect="equal")

    for i in range(len(order)):
        for j in range(len(order)):
            val = corr_sorted.iloc[i, j]
            color = "white" if val < 0.5 or val > 0.9 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color, fontweight="bold")

    short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ") for s in order]
    ax.set_xticks(range(len(order))); ax.set_yticks(range(len(order)))
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(short, fontsize=7)
    ax.set_title(fwilabel, fontsize=11)

    pc1 = fwi_results[fwicol]["loadings"][fwi_results[fwicol]["loadings"]["component"]=="PC1"]["explained_variance_ratio"].iloc[0]
    ax.text(0.02, 0.02, f"PC1 = {pc1:.1%}", transform=ax.transAxes, fontsize=7, color="0.3", style="italic")

fig.colorbar(im, ax=axes, shrink=0.5, label="Pearson r")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUTPUT_DIR / "redundancy_fwi_correlation_grid.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_fwi_correlation_grid.png")

/tmp/ipykernel_215571/1408503843.py:32: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.96])
findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=7.0, and fallback to the default font was disabled

In [24]:
# --- Visual: FWI Dendrogram Grid (2x3) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("FWI Component Dendrograms (distance = 1 - r)", fontsize=14, fontweight="bold", y=0.99)

for idx, (fwicol, fwilabel) in enumerate(FWI_COMPONENTS.items()):
    ax = axes[idx // 3][idx % 3]
    corr = fwi_results[fwicol]["corr"]
    distance = 1.0 - corr
    dist_condensed = squareform(distance.values, checks=False)
    Z = linkage(dist_condensed, method="average", metric="euclidean")
    short_labels = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ") for s in corr.columns]
    dendrogram(Z, labels=short_labels, ax=ax, leaf_rotation=45, leaf_font_size=7, color_threshold=0.0)
    ax.set_title(fwilabel, fontsize=11)
    ax.set_ylabel("Distance", fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUTPUT_DIR / "redundancy_fwi_dendrograms.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_fwi_dendrograms.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=7.0, and fallback to the default font was disabled

In [25]:
# --- Visual: FWI Recommendation Matrix ---
peinp_stations = [s for s in STATIONS if s != "stanhope"]
rec_color_map = {"keep": "#2ca02c", "defer": "#ff7f0e", "remove": "#d62728"}

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("FWI Redundancy Recommendations by Station and Component", fontsize=13, fontweight="bold")

for i, station in enumerate(peinp_stations):
    for j, (fwicol, fwilabel) in enumerate(FWI_COMPONENTS.items()):
        recs = fwi_results[fwicol]["recs"]
        row = recs[recs["station"] == station]
        if len(row) == 0:
            rec = "defer"
            corr_val = np.nan
        else:
            rec = row.iloc[0]["recommendation"]
            bench = fwi_results[fwicol]["bench"]
            bench_row = bench[bench["station"] == station]
            corr_val = bench_row.iloc[0]["correlation"] if len(bench_row) > 0 else np.nan

        color = rec_color_map[rec]
        ax.barh(i, 1, left=j, color=color, edgecolor="white", linewidth=1.5, height=0.7)
        txt = rec.upper()
        if not np.isnan(corr_val):
            txt += f"\nr={corr_val:.2f}"
        ax.text(j + 0.5, i, txt, ha="center", va="center", fontsize=7,
                fontweight="bold", color="white" if rec == "remove" else "black")

ax.set_yticks(range(len(peinp_stations)))
short_y = [s.replace("north_", "N. ").replace("stanley_", "S. ").replace("_", " ").title() for s in peinp_stations]
ax.set_yticklabels(short_y)
ax.set_xticks([j + 0.5 for j in range(len(FWI_COMPONENTS))])
ax.set_xticklabels(list(FWI_COMPONENTS.values()), rotation=0, fontsize=9)
ax.set_xlim(0, len(FWI_COMPONENTS))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=rec_color_map[k], edgecolor="white", label=k.capitalize())
                   for k in ["keep", "defer", "remove"]]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9, title="Recommendation")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_fwi_recommendations.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_fwi_recommendations.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=9.0, and fallback to the default font was disabled

## 12. Leave-One-Out FWI Degradation

The most direct redundancy test: identify which rows in each station's
hourly data were cross-station imputed from a given donor, then measure
how much FWI changes when that donor is removed.  Stations whose removal
causes < 1 FWI point MAE across the network are redundant.

In [26]:
# --- Identify imputed rows and their donors ---
donor_traces = {}
qf_patterns = {
    "air_temperature_c": ("air_temperature_c_qf", "air_temperature_c_src"),
    "relative_humidity_pct": ("relative_humidity_pct_qf", "relative_humidity_pct_src"),
    "wind_speed_kmh": ("wind_speed_kmh_qf", "wind_speed_kmh_src"),
}

for station in STATIONS:
    f = Path("data/processed") / station / "station_hourly.csv"
    df = pd.read_csv(f, parse_dates=["timestamp_utc"], low_memory=False)
    traces = {"station": station, "total_rows": len(df)}

    for fwicol in ["ffmc", "dmc", "dc", "isi", "bui", "fwi"]:
        if fwicol in df.columns:
            traces[f"{fwicol}_non_null"] = df[fwicol].notna().sum()
            traces[f"{fwicol}_pct"] = df[fwicol].notna().mean() * 100

    for var, (qf_col, src_col) in qf_patterns.items():
        if qf_col in df.columns:
            n_imputed = (df[qf_col] > 0).sum()
            traces[f"{var}_imputed"] = n_imputed
            if src_col in df.columns:
                src_counts = df.loc[df[qf_col] > 0, src_col].value_counts().to_dict()
                for src, cnt in src_counts.items():
                    traces[f"donor_{src}"] = cnt
        else:
            traces[f"{var}_imputed"] = 0

    donor_traces[station] = traces

trace_df = pd.DataFrame(donor_traces).T
print("Donor trace summary:")
print(trace_df[["total_rows", "fwi_non_null", "fwi_pct",
                "air_temperature_c_imputed", "wind_speed_kmh_imputed",
                "relative_humidity_pct_imputed"]].to_string())

Donor trace summary:
               total_rows fwi_non_null    fwi_pct air_temperature_c_imputed wind_speed_kmh_imputed relative_humidity_pct_imputed
cavendish           26092        22193  85.056722                         0                   3890                             0
greenwich           26092        21470  82.285758                     10158                   3249                          4229
north_rustico       26113        24947  95.534791                         0                   1188                             0
stanhope            24144        23081   95.59725                         0                      0                             0
stanley_bridge      26092        21563  82.642189                      4486                   8371                         26092
tracadie            23956        20771  86.704792                      3076                   3809                         23956


In [27]:
# --- Leave-One-Out: measure donor impact on each target ---
peinp_stations = [s for s in STATIONS if s != "stanhope"]
loo_results = []

for removed_station in peinp_stations:
    print(f"--- Removing: {removed_station} ---")

    for target_station in STATIONS:
        if target_station == removed_station:
            continue

        f = Path("data/processed") / target_station / "station_hourly.csv"
        df = pd.read_csv(f, parse_dates=["timestamp_utc"], low_memory=False)

        src_cols = [c for c in df.columns if c.endswith("_src")]
        mask = pd.Series(False, index=df.index)
        for sc in src_cols:
            mask |= df[sc].astype(str).str.contains(removed_station, na=False)

        n_affected = mask.sum()

        if n_affected > 0 and "fwi" in df.columns:
            baseline_vals = df.loc[mask, "fwi"].dropna()
            mae_contribution = baseline_vals.abs().mean() if len(baseline_vals) > 0 else 0.0
            loo_results.append({
                "removed": removed_station,
                "target": target_station,
                "n_affected_rows": n_affected,
                "n_affected_fwi": len(baseline_vals),
                "mae_upper_bound": mae_contribution,
            })
            print(f"  -> {target_station}: {n_affected} rows, {len(baseline_vals)} with FWI, MAE upper={mae_contribution:.2f}")
        else:
            loo_results.append({
                "removed": removed_station,
                "target": target_station,
                "n_affected_rows": n_affected,
                "n_affected_fwi": 0,
                "mae_upper_bound": 0.0,
            })

loo_df = pd.DataFrame(loo_results)
print("\nLeave-one-out summary:")
print(loo_df.to_string())

--- Removing: cavendish ---


  -> north_rustico: 2 rows, 0 with FWI, MAE upper=0.00


  -> stanley_bridge: 24600 rows, 20072 with FWI, MAE upper=2.65


  -> tracadie: 18672 rows, 18559 with FWI, MAE upper=3.57
--- Removing: greenwich ---


--- Removing: north_rustico ---
  -> cavendish: 3890 rows, 104 with FWI, MAE upper=0.41
  -> greenwich: 5 rows, 2 with FWI, MAE upper=7.72


  -> stanley_bridge: 3789 rows, 2211 with FWI, MAE upper=0.58
  -> tracadie: 5938 rows, 2827 with FWI, MAE upper=0.83
--- Removing: stanley_bridge ---


--- Removing: tracadie ---



Leave-one-out summary:
           removed          target  n_affected_rows  n_affected_fwi  mae_upper_bound
0        cavendish       greenwich                0               0         0.000000
1        cavendish   north_rustico                2               0         0.000000
2        cavendish        stanhope                0               0         0.000000
3        cavendish  stanley_bridge            24600           20072         2.647056
4        cavendish        tracadie            18672           18559         3.571283
5        greenwich       cavendish                0               0         0.000000
6        greenwich   north_rustico                0               0         0.000000
7        greenwich        stanhope                0               0         0.000000
8        greenwich  stanley_bridge                0               0         0.000000
9        greenwich        tracadie                0               0         0.000000
10   north_rustico       cavendish       

In [28]:
# --- Visual: LOO Degradation Heatmap ---
fig, ax = plt.subplots(figsize=(10, 7))
fig.suptitle("Leave-One-Out: Rows Affected When Donor Removed\n(hourly rows imputed from each donor at each target)",
             fontsize=12, fontweight="bold")

pivot = loo_df.pivot_table(index="removed", columns="target", values="n_affected_rows", fill_value=0)
for s in STATIONS:
    if s not in pivot.index:
        pivot.loc[s] = 0
    if s not in pivot.columns:
        pivot[s] = 0
pivot = pivot.sort_index(axis=0).sort_index(axis=1)

short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ").title() for s in pivot.index]

cmap = LinearSegmentedColormap.from_list("affected", ["#f7fbff", "#4292c6", "#08306b"], N=256)
im = ax.imshow(pivot.values, cmap=cmap, aspect="equal")

for i in range(len(pivot)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = "white" if val > pivot.values.max() * 0.5 else "black"
        ax.text(j, i, f"{int(val):,}", ha="center", va="center", fontsize=9, color=color, fontweight="bold")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ").title() for s in pivot.columns],
                    rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(short, fontsize=9)
ax.set_xlabel("Target Station")
ax.set_ylabel("Removed Donor")
fig.colorbar(im, ax=ax, shrink=0.8, label="Affected Rows")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_loo_degradation.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_loo_degradation.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=9.0, and fallback to the default font was disabled

In [29]:
# --- Visual: LOO Summary Bar Chart ---
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle("Leave-One-Out: Total Network Impact per Removed Station",
             fontsize=13, fontweight="bold")

summary = loo_df.groupby("removed").agg(
    total_affected=("n_affected_rows", "sum"),
    total_fwi_affected=("n_affected_fwi", "sum"),
    n_targets=("target", "count"),
    mean_mae=("mae_upper_bound", "mean"),
).sort_values("total_affected", ascending=True)

short = [s.replace("north_", "N.").replace("stanley_", "S.").replace("_", " ").title() for s in summary.index]
x = np.arange(len(summary))
w = 0.35

ax.barh(x - w/2, summary["total_affected"], w, label="Total affected rows", color="#4393c3")
ax.barh(x + w/2, summary["total_fwi_affected"], w, label="Affected rows with FWI", color="#d6604d")

ax.set_yticks(x)
ax.set_yticklabels(short, fontsize=10)
ax.set_xlabel("Number of Hourly Rows")
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.3)

for i, (idx_name, row) in enumerate(summary.iterrows()):
    ax.text(row["total_affected"] + 50, i - w/2, f"{int(row['total_affected']):,}", va="center", fontsize=8)
    ax.text(row["total_fwi_affected"] + 50, i + w/2, f"{int(row['total_fwi_affected']):,}", va="center", fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "redundancy_loo_summary.png", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: figures/redundancy_loo_summary.png")

findfont: Generic family 'sans-serif' not found because none of the following families were found: DejaVu Sans, Bitstream Vera Sans, Computer Modern Sans Serif, Lucida Grande, Verdana, Geneva, Lucid, Arial, Helvetica, Avant Garde, sans-serif


findfont: Font family ['DejaVu Sans'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

## 13. Output File Listing

All figures generated by this notebook.

In [30]:
from pathlib import Path
figs = sorted(OUTPUT_DIR.glob("redundancy_*.png"))
for f in figs:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:55s} {size_kb:>6.0f} KB")
print(f"\n  {len(figs)} figures total")

  redundancy_benchmark_bars.png                               92 KB
  redundancy_bootstrap_ci_forest.png                         217 KB
  redundancy_bootstrap_ci_heatmap.png                        130 KB
  redundancy_correlation_heatmaps.png                        243 KB
  redundancy_dendrograms.png                                 114 KB
  redundancy_extreme_degradation.png                         108 KB
  redundancy_extreme_vs_full.png                             183 KB
  redundancy_fire_weather_corr.png                           155 KB
  redundancy_fwi_correlation_grid.png                        262 KB
  redundancy_fwi_dendrograms.png                             125 KB
  redundancy_fwi_recommendations.png                          77 KB
  redundancy_loo_degradation.png                              90 KB
  redundancy_loo_summary.png                                  55 KB
  redundancy_partial_corr_scatter.png                        206 KB
  redundancy_recommendations.png                